# S1CogBench lite targets in Colab

This notebook runs a Hugging Face causal language model on the eight lite targets of [S1CogBench](https://github.com/dengzhe-hou/S1CogBench) and counts the targets on which it beats uniform choice with a 95% bootstrap interval. The model is read through the repository's [Hugging Face template](https://github.com/dengzhe-hou/S1CogBench/blob/main/examples/templates/hf_causal_lm_choice.py), which takes the probabilities of the legal option labels as the accepted Llama and Qwen rows do.

The lite count is a quick check, not the benchmark. The eight targets were chosen with the evaluated models, so a new model can rank differently on all 26 ([Reliability](https://github.com/dengzhe-hou/S1CogBench/blob/main/docs/RELIABILITY.md)).

Choose a GPU runtime first, under Runtime > Change runtime type.

## 1. Choose a model

Use a Hub id or a local directory, and add `@<commit>` to pin a revision. A gated model also needs the `HF_TOKEN` secret of step 4. The default fits on a T4. The pinned Transformers 4.57.1 does not know architectures newer than it, such as Qwen3.5. For those, run `!pip install -U transformers` after step 3.

In [ ]:
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

## 2. Check the runtime

The pinned NumPy 1.26 needs Python 3.11 or 3.12. If Colab has moved to a newer Python, pick an older runtime version under Runtime > Change runtime type. GPUs before Ampere, such as the T4, run the model in float16, and newer ones in bfloat16.

In [ ]:
import os
import shutil
import subprocess
import sys

assert sys.version_info[:2] in ((3, 11), (3, 12)), (
    f'Python {sys.version.split()[0]} is too new for the pinned NumPy 1.26. '
    'Pick a runtime version with Python 3.12 under Runtime > Change runtime type.')
assert shutil.which('nvidia-smi'), 'No GPU found. Choose a GPU runtime under Runtime > Change runtime type.'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                     capture_output=True, text=True, check=True).stdout.splitlines()[0]
name, capability = (x.strip() for x in gpu.split(','))
DTYPE = 'bfloat16' if float(capability) >= 8 else 'float16'
print(f'{name}, compute capability {capability}, dtype {DTYPE}')

## 3. Install

This keeps Colab's CUDA build of PyTorch and installs the other pins of `requirements.txt`, whose PyTorch line is the CPU build. If pip suggests restarting the session, you can skip that, because every later step runs in its own process.

In [ ]:
if not os.path.isdir('/content/S1CogBench'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/dengzhe-hou/S1CogBench.git', '/content/S1CogBench'], check=True)
%cd /content/S1CogBench
!git log --oneline -1
!grep -v -E '^(torch==|--extra-index-url)' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 4. Get the data

The lite targets read four public sources at their pinned revisions.

In [ ]:
!for s in cogbench centaur_evaluation coggym tomi; do python scripts/fetch_public_sources.py $s; done

Multi-attribute choice also needs Psych-101-test, which is gated. To include it, accept the conditions of [marcelbinz/Psych-101-test](https://huggingface.co/datasets/marcelbinz/Psych-101-test) on Hugging Face (approval is automatic), add a read token as the Colab secret `HF_TOKEN` with notebook access, and run the next cell. Otherwise skip the cell, and `score` lists that target as missing. Do not share the downloaded file or the per-item outputs of that mode, which contain participant responses.

In [ ]:
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!python scripts/fetch_public_sources.py psych101

In [ ]:
!python -m s1cogbench doctor --tasks lite

## 5. Run and score

Each mode runs through the original runner and scorer of the paper. Nox (4B) needed about 50 minutes on an A100 MIG slice, and the time grows with model size. `--output` must be a new directory, so change `OUT` before running again.

In [ ]:
OUT = 'runs/' + MODEL.split('@')[0].rstrip('/').split('/')[-1] + '-lite'
!S1_HF_DTYPE={DTYPE} python -m s1cogbench eval --adapter hf:{MODEL} --device cuda:0 --tasks lite --output {OUT}

In [ ]:
!python -m s1cogbench score {OUT}

## 6. Compare

`score` writes `reliability.json` into the output folder, next to `results.json`. The published lite counts (of 8) are 1 for Laya, 7 for Intern-Decision-4B, 3 for Llama3.1-8B, 4 for Minitaur8B, 1 for Qwen3.5-4B, 8 for Decision-2.0-Nox-4B and 2 for CLM-v0.1-8B.

To compare target by target, download `results.json` and open it with Add your results on the [project page](https://dengzhe-hou.github.io/S1CogBench/). The [quickstart](https://github.com/dengzhe-hou/S1CogBench/blob/main/docs/QUICKSTART.md) covers the full suite and the community leaderboard.

In [ ]:
from google.colab import files

files.download(f'{OUT}/results.json')